# FairyZero — đo hiệu năng sinh dữ liệu (Colab)

Chạy lần lượt từ trên xuống. **Cell 6 là phần đo**; nó in ra một bảng —
copy nguyên bảng đó gửi lại.

Dùng nhánh `mcts-capacity-256`: luật mới (Hậu thay Amazon, 8-checks) + bộ đo
mới (NN eval/giây, batch trung bình mỗi Run, % phí do padding).

Tổng thời gian ≈ 40 phút: build ~10, kiểm tra ~2, đo ~25.


## 1. Kiểm tra GPU

Nếu báo `command not found` thì chưa bật GPU:
**Runtime → Change runtime type → T4 GPU**.


In [ ]:
!nvidia-smi


## 2. Lấy mã nguồn mới nhất


In [ ]:
%cd /content
!rm -rf chess_variant_engine
!git clone -q --depth 1 -b mcts-capacity-256 \
    https://github.com/phuc11731510/chess_variant_engine.git
!git -C chess_variant_engine log -1 --oneline


## 3. Build (≈8-12 phút)

Tải ONNX Runtime GPU rồi build với `-Duse_cuda=true`.

**Bắt buộc build lại** — bản `.rar` trên Releases là luật CŨ (Amazon, 7-checks).


In [ ]:
!bash /content/chess_variant_engine/custom_engine/scripts/colab_setup.sh


## 4. Tạo `run.sh`

Mỗi cell Colab là một shell mới nên `LD_LIBRARY_PATH` mà cell 3 đặt không sống
sót sang cell sau. `colab_prebuilt.sh wrap` sinh ra `run.sh` tự đặt lại đường
dẫn thư viện rồi gọi engine. Phải thấy dòng `[prebuilt] OK`.


In [ ]:
!bash /content/chess_variant_engine/custom_engine/scripts/colab_prebuilt.sh wrap


## 5. Kiểm tra đúng đắn trước khi đo

Xác nhận bản build trên Colab chạy đúng luật mới. Phải thấy **toàn `[PASS]`**.
`--test-perft` là quan trọng nhất: nó đối chiếu sinh nước đi với
Fairy-Stockfish gốc ở mọi độ sâu.

Nếu có `[FAIL]` thì **dừng lại**, gửi tôi xem trước khi đo.


In [ ]:
%cd /content/chess_variant_engine/custom_engine
for t in ['--test-adapter','--test-perft','--test-rules',
          '--test-encoder','--test-trainingdata','--test-policy']:
    print(f'{t:22}', end='')
    !bash run.sh {t} 2>&1 | grep -E '^\[PASS\]|^\[FAIL\]' | tail -1


## 6. Lấy mạng để đo

Dùng mạng gen-12 cũ. Nó học luật cũ nên chơi không hay, nhưng **kiến trúc
giống hệt** (12×144, 226 plane) nên tốc độ đo được vẫn đại diện. Điều quan
trọng là mọi cấu hình dùng **chung một mạng**.


In [ ]:
%cd /content
!wget -q https://github.com/phuc11731510/chess_variant_engine/releases/download/v1.0.0/12bx144fx8s_12.onnx
!ls -la 12bx144fx8s_12.onnx


## 7. ĐO — quét 6 cấu hình  ⬅ PHẦN CHÍNH

Mỗi cấu hình 240 giây → ≈25 phút. Muốn nhanh hơn: đổi `SECS=240` thành `SECS=120`.

| | parallel | fixed-batch | aggregate | để trả lời câu hỏi |
|---|---|---|---|---|
| A | 4 | 16 | tắt | **mốc nền** — cấu hình bạn đang dùng |
| B | 1 | 16 | tắt | chạy song song có giúp gì không |
| C | 4 | 64 | bật | gom batch có hiệu quả không |
| D | 8 | 64 | bật | thêm producer thì sao |
| E | 16 | 64 | bật | nhiều producer, timeout ngắn |
| F | 32 | 64 | bật | đẩy tới giới hạn |

Script tự lấy mẫu `sm%` bằng `nvidia-smi` song song, và tự xoá `.gz` sau mỗi
lượt (dữ liệu benchmark không dùng để huấn luyện).


In [ ]:
%cd /content
!WEIGHTS=/content/12bx144fx8s_12.onnx \
 ENGINE=/content/chess_variant_engine/custom_engine/run.sh \
 SECS=240 \
 bash /content/chess_variant_engine/custom_engine/scripts/colab_benchmark.sh


## 8. Gửi lại cho tôi

Copy **toàn bộ khối `KET QUA`** ở cuối cell 7 (phần "Cách đọc" bỏ được).

Nếu cell nào lỗi thì gửi luôn dòng lỗi — đừng bỏ qua rồi chạy tiếp.


## 9. (Tuỳ chọn) Đóng mắt xích cuối chưa kiểm của pipeline huấn luyện

Không liên quan tới tốc độ, nhưng rẻ. Sinh một ít dữ liệu **luật mới** rồi
chạy `train.py` 5 bước để chắc chắn không vỡ shape ở đâu.

Thấy loss in ra là xong (giá trị bao nhiêu không quan trọng).


In [ ]:
%cd /content/chess_variant_engine/custom_engine
!bash run.sh --selfplay --games 3 --visits 100 --max-moves 200 \
    --parallel 2 --provider cuda --fixed-batch 16 \
    --weights /content/12bx144fx8s_12.onnx --out /content/smoke 2>&1 | tail -4
!python python/train.py --data /content/smoke --epochs 1 --max-steps 5 \
    --batch 8 --channels 144 --blocks 12 --report-every 1 \
    --out /content/smoke.onnx 2>&1 | tail -20


## 10. (Tuỳ chọn) Lưu bản build lên Drive

Để phiên Colab sau **bỏ qua 10 phút build**. Cần mount Drive trước.
Phiên sau chỉ cần chạy cell 2 rồi `colab_prebuilt.sh restore`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!bash /content/chess_variant_engine/custom_engine/scripts/colab_prebuilt.sh save
